In [1]:
import pandas as pd
import numpy as np
import sys, os
from functools import reduce
import pyspark
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
import pyspark.sql.types as T
import pyspark.sql.functions as F

In [2]:
# test if the script is running as a standalone program or a notebook
if __name__ == '__main__' and '__file__' in globals():
    print("Running as a script")
    application_name = sys.argv[1]
    s3a_access_key = sys.argv[2]
    s3a_secret_key = sys.argv[3]
    input_file_1 = sys.argv[4]
    output_file_1 = sys.argv[5]
    interactive = False
else:
    interactive = True
    print("Running as a notebook")
    pv = !{sys.executable} --version
    print("Python version ", pv)
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    print("PySpark version ", pyspark.__version__)
    jv = !java --version
    print("Java version ", jv)
    # defaults so the cells below run without command-line arguments
    s3a_access_key = os.environ.get("AWS_ACCESS_KEY_ID", "")
    s3a_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY", "")
    input_file_1 = os.path.join(os.getcwd(), "data/US-Constitution.txt")
    output_file_1 = os.path.join(os.getcwd(), "tmp/word_count_result")

Running as a notebook
Python version  ['Python 3.13.9']
PySpark version  4.2.0
Java version  ['openjdk 17.0.20 2026-07-21', 'OpenJDK Runtime Environment Temurin-17.0.20+8 (build 17.0.20+8)', 'OpenJDK 64-Bit Server VM Temurin-17.0.20+8 (build 17.0.20+8, mixed mode, sharing)']


In [4]:
spark = (SparkSession.builder.appName("Word Count Example")
         ##.config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
         #.config('spark.hadoop.fs.s3a.access.key', s3a_access_key)
         #.config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
         #.config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-2.amazonaws.com")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/02 12:47:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
poem = """One fish Two fish Red fish Blue fish
Black fish Blue fish Old fish New fish
This one has a little car
This one has a little star"""

In [8]:
lines = poem.split()
counts = {}
for line in lines:
    words = line.split()
    for word in words:
        if word in counts:
            counts[word] += 1
        else:
            counts[word]=1
sorted(counts.items(), key=lambda r: r[1], reverse=True)

[('fish', 8),
 ('Blue', 2),
 ('This', 2),
 ('one', 2),
 ('has', 2),
 ('a', 2),
 ('little', 2),
 ('One', 1),
 ('Two', 1),
 ('Red', 1),
 ('Black', 1),
 ('Old', 1),
 ('New', 1),
 ('car', 1),
 ('star', 1)]

In [9]:
from collections import Counter
Counter(poem.split())

Counter({'fish': 8,
         'Blue': 2,
         'This': 2,
         'one': 2,
         'has': 2,
         'a': 2,
         'little': 2,
         'One': 1,
         'Two': 1,
         'Red': 1,
         'Black': 1,
         'Old': 1,
         'New': 1,
         'car': 1,
         'star': 1})

# Distributed solution

In [18]:
lines = sc.textFile("/Users/hanisaf/BigData/enwik9")
lines.count()
#poem.splitlines()
#len(lines)

13147026

In [21]:
counts = ( 
    lines
    .flatMap(lambda line: line.split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda x, y: x + y)
    .sortBy(lambda t: t[1], ascending=False)
)
counts.saveAsTextFile('counts.txt')

## Explanation

`Counter(poem.split())` above is the single-machine version: one process, one pass, the
whole text in memory. It is the right tool until the text stops fitting on one machine.

The rest of this notebook rebuilds the same computation with **Spark RDDs only** (no
DataFrames, no SQL), following the six stages from the slide deck:

| Slide stage | What happens | RDD operation |
|---|---|---|
| **Input** | the text as a whole | `sc.parallelize(...)` / `sc.textFile(...)` |
| **Splitting** | the input is cut into independent pieces | *partitions* -- one per task |
| **Mapping** | each piece emits `(word, 1)` pairs, in parallel, with no coordination | `flatMap` + `map` |
| **Shuffling** | pairs are moved so that all copies of a key end up on the same machine | `partitionBy` / the shuffle inside `reduceByKey` |
| **Reducing** | the values for each key are summed | `reduceByKey(add)` |
| **Final result** | the counts are returned or written out | `collect()` / `saveAsTextFile()` |

We use the same three lines as the slide (`Deer Bear River`, ...) so the numbers at the
end should match the diagram exactly, then repeat the exercise on real text.

### A helper for looking inside the RDD

Nothing in a Spark job is visible by default -- transformations are lazy and the data
lives on the executors. `glom()` collapses each partition into a list, which lets us
print the contents **partition by partition**. That is what makes the stages of the
diagram visible.

This is a teaching device, not something to do in production: `glom().collect()` brings
everything to the driver, and each call re-runs the whole lineage from the start.

In [12]:
from operator import add          # the reduce function used below


def show(rdd, label, limit=8):
    """Print an RDD partition by partition (teaching helper -- triggers a job)."""
    parts = rdd.glom().collect()
    total = sum(len(p) for p in parts)
    print(f"=== {label} === {rdd.getNumPartitions()} partitions, {total} records")
    for i, p in enumerate(parts):
        tail = "" if len(p) <= limit else f"   ... (+{len(p) - limit} more)"
        print(f"  partition {i}: {p[:limit]}{tail}")
    print()

## Stages 1-2: Input and Splitting

`sc.parallelize(lines, 3)` takes a local list and cuts it into 3 partitions -- the
"Splitting" column of the diagram. A partition is the unit of parallelism: one partition
is processed by one task on one core, independently of all the others.

With a real file, `sc.textFile(path)` does the same thing but the splits come from the
file blocks (128 MB by default on HDFS/S3). That is the **data locality** idea from the
deck: Spark tries to run each task on the machine that already holds its block, so the
input never crosses the network -- only the shuffle does.

In [13]:
lines = poem.splitlines()

input_rdd = sc.parallelize(lines, 10)          # 3 partitions = the 3 "splits" on the slide
show(input_rdd, "INPUT / SPLITTING")

=== INPUT / SPLITTING === 10 partitions, 4 records
  partition 0: []
  partition 1: []
  partition 2: ['One fish Two fish Red fish Blue fish']
  partition 3: []
  partition 4: ['Black fish Blue fish Old fish New fish']
  partition 5: []
  partition 6: []
  partition 7: ['This one has a little car']
  partition 8: []
  partition 9: ['This one has a little star']



## Stage 3: Mapping

Two narrow transformations, both running inside the partition they read from -- no data
moves, and each task can finish without waiting for any other:

- `flatMap(lambda line: line.split())` -- one line in, *many* words out (this is what
  makes it `flatMap` rather than `map`).
- `map(lambda w: (w.lower(), 1))` -- tag every occurrence with the count 1. Lowercasing
  here is the "normalisation" step; do it before the shuffle so `Deer` and `deer` are
  treated as the same key.

Notice the record count grows from 3 to 9 while the number of partitions stays at 3.

In [14]:
words = input_rdd.flatMap(lambda line: line.split())
show(words, "MAPPING (a) -- flatMap: line -> words")



=== MAPPING (a) -- flatMap: line -> words === 10 partitions, 28 records
  partition 0: []
  partition 1: []
  partition 2: ['One', 'fish', 'Two', 'fish', 'Red', 'fish', 'Blue', 'fish']
  partition 3: []
  partition 4: ['Black', 'fish', 'Blue', 'fish', 'Old', 'fish', 'New', 'fish']
  partition 5: []
  partition 6: []
  partition 7: ['This', 'one', 'has', 'a', 'little', 'car']
  partition 8: []
  partition 9: ['This', 'one', 'has', 'a', 'little', 'star']



In [15]:
pairs = words.map(lambda w: (w.lower(), 1))
show(pairs, "MAPPING (b) -- map: word -> (word, 1)")

=== MAPPING (b) -- map: word -> (word, 1) === 10 partitions, 28 records
  partition 0: []
  partition 1: []
  partition 2: [('one', 1), ('fish', 1), ('two', 1), ('fish', 1), ('red', 1), ('fish', 1), ('blue', 1), ('fish', 1)]
  partition 3: []
  partition 4: [('black', 1), ('fish', 1), ('blue', 1), ('fish', 1), ('old', 1), ('fish', 1), ('new', 1), ('fish', 1)]
  partition 5: []
  partition 6: []
  partition 7: [('this', 1), ('one', 1), ('has', 1), ('a', 1), ('little', 1), ('car', 1)]
  partition 8: []
  partition 9: [('this', 1), ('one', 1), ('has', 1), ('a', 1), ('little', 1), ('star', 1)]



## Stage 4: Shuffling

This is the only stage where data crosses the network. To sum the counts for `car`, every
`("car", 1)` has to end up in the same place, and they are currently spread over three
partitions.

Spark decides the destination with a **hash partitioner**:

```
target_partition = hash(key) % numPartitions
```

The same key always hashes to the same partition, so all its values meet. `partitionBy(3)`
below performs that redistribution and nothing else, which makes it the clearest way to
*see* a shuffle. Compare the partition contents with the mapping stage above: the records
are the same, their location is not.

Two consequences worth remembering:

- A shuffle is a **wide dependency**: an output partition depends on *all* input
  partitions. Everything before it must finish before anything after it can start, which
  is exactly what defines a **stage boundary**.
- Partitions are rarely balanced. Here two keys hash to the same partition and the other
  two get one each, so one task receives more than twice the records of another. Real
  datasets skew far worse -- one very common key can hold up an entire stage.

In [16]:
shuffled = pairs.partitionBy(10)          # hash(key) % 3 decides the destination
show(shuffled, "SHUFFLING -- after partitionBy(3)")

# which key ended up where?
placement = shuffled.mapPartitionsWithIndex(
    lambda i, it: [(k, i) for k, _ in it]).distinct().collect()
print("key -> partition:", dict(sorted(placement)))

=== SHUFFLING -- after partitionBy(3) === 10 partitions, 28 records
  partition 0: [('little', 1), ('little', 1), ('star', 1)]
  partition 1: []
  partition 2: [('blue', 1), ('blue', 1), ('old', 1)]
  partition 3: []
  partition 4: [('red', 1), ('black', 1)]
  partition 5: [('one', 1), ('two', 1), ('one', 1), ('one', 1)]
  partition 6: []
  partition 7: [('fish', 1), ('fish', 1), ('fish', 1), ('fish', 1), ('fish', 1), ('fish', 1), ('fish', 1), ('fish', 1)]
  partition 8: [('new', 1), ('this', 1), ('car', 1), ('this', 1)]
  partition 9: [('has', 1), ('a', 1), ('has', 1), ('a', 1)]

key -> partition: {'a': 9, 'black': 4, 'blue': 2, 'car': 8, 'fish': 7, 'has': 9, 'little': 0, 'new': 8, 'old': 2, 'one': 5, 'red': 4, 'star': 0, 'this': 8, 'two': 5}


### The combiner: aggregate before you shuffle

Sending nine `(word, 1)` records over the network to add up nine ones is wasteful. Hadoop
calls the fix a **combiner**: run a partial reduce on the map side first, so each
partition sends at most one record per distinct word.

The cell below does it by hand with `mapPartitions` to make it visible. You would not
normally write this -- `reduceByKey` does it for you automatically, and that is the main
reason to prefer it over `groupByKey`.

In [15]:
from collections import Counter

combined = pairs.mapPartitions(lambda it: iter(Counter(k for k, _ in it).items()))
show(combined, "MAP-SIDE COMBINE (what reduceByKey does for free)")

before = pairs.mapPartitions(lambda it: [sum(1 for _ in it)]).collect()
after  = combined.mapPartitions(lambda it: [sum(1 for _ in it)]).collect()
print("records per partition before combine:", before, "->", sum(before), "total")
print("records per partition after  combine:", after, "->", sum(after), "total")

=== MAP-SIDE COMBINE (what reduceByKey does for free) === 10 partitions, 127 records
  partition 0: [('beautiful', 1), ('is', 1), ('better', 1), ('than', 1), ('ugly.', 1)]
  partition 1: [('explicit', 1), ('is', 2), ('better', 2), ('than', 2), ('implicit.', 1), ('simple', 1), ('complex.', 1)]
  partition 2: [('complex', 1), ('is', 2), ('better', 2), ('than', 2), ('complicated.', 1), ('flat', 1), ('nested.', 1)]
  partition 3: [('sparse', 1), ('is', 1), ('better', 1), ('than', 1), ('dense.', 1), ('readability', 1), ('counts.', 1)]
  partition 4: [('special', 2), ('cases', 1), ("aren't", 1), ('enough', 1), ('to', 1), ('break', 1), ('the', 1), ('rules.', 1)]   ... (+4 more)
  partition 5: [('errors', 1), ('should', 1), ('never', 1), ('pass', 1), ('silently.', 1), ('unless', 1), ('explicitly', 1), ('silenced.', 1)]
  partition 6: [('in', 1), ('the', 2), ('face', 1), ('of', 1), ('ambiguity,', 1), ('refuse', 1), ('temptation', 1), ('to', 2)]   ... (+13 more)
  partition 7: [('although', 1), 

## Stages 5-6: Reducing and the final result

`reduceByKey(add)` is the whole map-side-combine + shuffle + reduce sequence in one call:
it aggregates locally, shuffles the partial counts, then aggregates again per key.

`reduceByKey` is still a transformation -- lazy. The job only runs when `collect()` (an
action) asks for the result. The counts below should match the diagram on the slide:
Bear 2, Car 3, Deer 2, River 2.

In [16]:
counts = pairs.reduceByKey(add)
show(counts, "REDUCING -- reduceByKey(add)")

result = sorted(counts.collect(), key=lambda kv: (-kv[1], kv[0]))
print("FINAL RESULT")
for word, n in result:
    print(f"  {word:8s} {n}")

=== REDUCING -- reduceByKey(add) === 10 partitions, 88 records
  partition 0: [('ugly.', 1), ('implicit.', 1), ('flat', 1), ('enough', 1), ('practicality', 1), ('should', 2), ('unless', 2), ('explicitly', 1)]   ... (+4 more)
  partition 1: [('better', 8), ('simple', 1), ('beats', 1), ('in', 1), ('face', 1), ('refuse', 1), ('one--', 1), ('that', 1)]   ... (+1 more)
  partition 2: [('than', 8), ('to', 5), ('rules.', 1), ('although', 3), ('errors', 1), ('first', 1), ('easy', 1), ('more', 1)]
  partition 3: [('beautiful', 1), ('complex', 1), ('only', 1), ("you're", 1), ('hard', 1), ('--', 1), ('those!', 1)]
  partition 4: [('sparse', 1), ('purity.', 1), ('never', 2), ('silenced.', 1), ('ambiguity,', 1), ('idea.', 2), ('are', 1), ("let's", 1)]
  partition 5: [('is', 10), ('complex.', 1), ('complicated.', 1), ('nested.', 1), ('break', 1), ('temptation', 1), ('be', 3), ('one', 2)]   ... (+8 more)
  partition 6: [('explicit', 1), ('guess.', 1), ('preferably', 1), ('often', 1), ('good', 1), ('n

## Reading the DAG: where are the stages?

`toDebugString()` prints the lineage bottom-up. The two things to look for:

- **`ShuffledRDD`** -- the shuffle itself.
- **The `+-` indentation** -- every `+-(n)` marks a **stage boundary**; `n` is the number
  of tasks in that stage. Everything at the same indentation level is one stage, pipelined
  into a single pass over the data.

So this job is 2 stages: *(map side)* read -> flatMap -> map -> local combine -> write
shuffle files, then *(reduce side)* fetch -> sum -> collect. You can see the same split in
the Spark UI under the **Stages** tab while the job runs.

In [17]:
print(counts.toDebugString().decode())

(10) PythonRDD[34] at collect at /tmp/ipykernel_2116/4129150726.py:4 []
 |   MapPartitionsRDD[32] at mapPartitions at PythonRDD.scala:170 []
 |   ShuffledRDD[31] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(10) PairwiseRDD[30] at reduceByKey at /tmp/ipykernel_2116/4129150726.py:1 []
    |   PythonRDD[29] at reduceByKey at /tmp/ipykernel_2116/4129150726.py:1 []
    |   ParallelCollectionRDD[12] at readRDDFromFile at PythonRDD.scala:299 []


### `reduceByKey` vs `groupByKey`

Both produce the right answer and both shuffle, but they shuffle very different amounts:

- `groupByKey()` moves **every individual record**, then builds a list per key on the
  reduce side -- and that list must fit in the memory of one executor.
- `reduceByKey()` combines locally first, so it moves **one record per distinct word per
  partition**.

On three lines the difference is trivial. The cell below measures it on the Zen poem, and
the gap widens with every duplicate word -- on a real corpus it is the difference between
a job that runs and one that dies with an out-of-memory error.

In [18]:
poem_pairs = (sc.parallelize(poem.splitlines(), 4)
              .flatMap(lambda line: line.split())
              .map(lambda w: (w.lower(), 1)))

n_raw = poem_pairs.count()
n_combined = poem_pairs.mapPartitions(
    lambda it: iter(Counter(k for k, _ in it).items())).count()

print(f"groupByKey  would shuffle {n_raw} records")
print(f"reduceByKey shuffles      {n_combined} records "
      f"({100 * (1 - n_combined / n_raw):.0f}% less)")

# same answer, different cost
print()
print(poem_pairs.groupByKey().mapValues(sum).toDebugString().decode())

groupByKey  would shuffle 137 records
reduceByKey shuffles      103 records (25% less)

(4) PythonRDD[42] at RDD at PythonRDD.scala:58 []
 |  MapPartitionsRDD[41] at mapPartitions at PythonRDD.scala:170 []
 |  ShuffledRDD[40] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(4) PairwiseRDD[39] at groupByKey at /tmp/ipykernel_2116/3033440301.py:15 []
    |  PythonRDD[38] at groupByKey at /tmp/ipykernel_2116/3033440301.py:15 []
    |  ParallelCollectionRDD[35] at readRDDFromFile at PythonRDD.scala:299 []


## Shutdown

In [19]:
spark.stop()